In [ ]:
import json # giúp python đọc và hieu được định dạng JSON trả về từ AI
import time # giúp thêm delay khi bị rate limit để tránh bị block liên tục
import os
import google.generativeai as genai
from dotenv import load_dotenv # giúp load biến môi trường từ file .env để bảo mật API Key
from pydantic import BaseModel, Field # Giúp định nghĩa khuôn dữ liệu và ép kiểu dữ liệu một cách dễ dàng, tránh lỗi khi lưu vào Firebase
# 1. Định nghĩa cái khuôn dữ liệu (Giữ nguyên để kiểm tra sau khi bóc tách)
class InvoiceStructuredOutput(BaseModel): # tạo ra khuôn và quy định kiểu dữ liệu
    vendor: str = Field(..., description="Tên nhà cung cấp, nếu có thể trích xuất được")
    date: str = Field(..., description="Ngày tháng trên hóa đơn, nếu có thể trích xuất được")
    total_amount: float = Field(..., description="Tổng số tiền trên hóa đơn, nếu có thể trích xuất được")
    currency: str = Field(..., description="Mã ISO của loại tiền tệ, nếu có thể trích xuất được")

load_dotenv()
MY_API_KEY = os.getenv("GOOGLE_API_KEY") #API Key
if not MY_API_KEY:
    raise ValueError("API Key not found. Please set GOOGLE_API_KEY in your .env file.")

genai.configure(api_key=MY_API_KEY)

def process_ocr_with_gemini(ocr_text: str) -> InvoiceStructuredOutput:
    # Ép con AI bằng Prompt cực kỳ nghiêm ngặt để nó nhả đúng cấu trúc JSON
    system_instruction = (
        "You are an expert financial AI specializing in multilingual invoice parsing (Asia & Europe).\n"
        "Your task is to analyze the raw OCR text and extract structured information strictly fitting the schema.\n\n"
    
        "DISAMBIGUATION RULES:\n"
        "1. CHINA vs TAIWAN: 400-hotline/0371 = CNY. 統一編號/02-hotline = TWD.\n"
        "2. Tax IDs: 'MST' -> VND | 'VAT Reg' -> GBP/EUR | '納稅人識別號' -> CNY\n"
        "3. If no indicators are found, fallback to 'VND'."
        
        "OUTPUT FORMAT (STRICT):\n"
        "You must respond ONLY with a valid JSON object. You are FORBIDDEN to use any keys other than the following 4 keys:\n"
        '1. "vendor" (string): The store name, You MUST fix OCR spelling errors, restore missing diacritics, and capitalize properly based on context.\n'
        '2. "date" (string): The date in YYYY-MM-DD format.\n'
        '3. "total_amount" (float): The final paid amount. MUST be a pure number without commas or currency symbols (e.g., 47000 or 47000.5).\n'
        '4. "currency" (string): The ISO currency code.\n'
        "Do not add extra keys like 'merchant_name', 'change_returned', or 'tax'."
    )
    try:
       model = genai.GenerativeModel(
            model_name="gemini-3.5-flash",
            system_instruction=system_instruction,
            # ✨ Ép Gemini nhả ra định dạng JSON
            generation_config={"response_mime_type": "application/json", "temperature": 0.0}
        )
       response = model.generate_content(f"Extract structured data from this OCR text:\n\n{ocr_text}")
       json_string = response.text.strip()
       parsed_data = json.loads(json_string)
       return InvoiceStructuredOutput(**parsed_data) 
    except Exception as e:
        print(f"Error processing OCR with SLM: {e}")
        return InvoiceStructuredOutput(vendor="UNKNOWN", date="UNKNOWN", total_amount=0.0, currency="VND") # Fallback cuối cùng nếu tất cả đều fail, trả về VND thay vì UNKNOWN để tránh lỗi khi lưu vào Firebase

In [ ]:
from pathlib import Path

input_path = Path('../Seg_OCR_Tri/outputs/OCR_RESULT.txt')
output_path = Path('./output/OCR_RESULT_notebook.json')

print('--- ĐANG ĐỌC OCR_RESULT.txt TỪ Seg_OCR_Tri/outputs ---')
if not input_path.exists():
    raise FileNotFoundError(f'Không tìm thấy file đầu vào: {input_path.resolve()}')

ocr_text = input_path.read_text(encoding='utf-8', errors='replace')
result = process_ocr_with_slm(ocr_text)

output_data = result.model_dump()
output_path.write_text(json.dumps(output_data, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Đã xuất JSON vào: {output_path.resolve()}')
print(json.dumps(output_data, ensure_ascii=False, indent=2))

In [32]:
# Giả lập một cục text cực kỳ lộn xộn, lỗi font, sai chính tả do tầng OCR quét từ ảnh sang
mock_ocr_output = mock_ocr_output_vietnam = """
--- OCR START ---
H0A D0N BAN Lf
ClKCLE K VlET NAM - Cua hang 125
DC: 125 CMT8, Phuong 5, TPHCM
MST : 0312345678-001
Ngay in: 15/06/2026 08:15:00
---------------------------------
Ten san pharn        SL    Thanh Tien
Sting Dau 330rnl     2     20,000
Banh Mi Cha Lua      1     15,000
M1 Xao HaoHao        1     12,000
---------------------------------
T0NG C0NG:                  47,000
Khuyen rnai:                0
Khach dua:                  50,000
Tien thoi:                  3,000
Cam on QlJy khach va hen gap lal!
--- OCR END ---
"""

print("--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---")

# Gọi hàm xử lý
result = process_ocr_with_slm(mock_ocr_output)

print("\n--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---")
# Dùng model_dump() thay cho dict() để không bị báo lỗi cảnh báo màu vàng nữa nhé bạn
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=4))

print("\n--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---")
print(f"Số tiền trích xuất: {result.total_amount} (Kiểu dữ liệu: {type(result.total_amount)})")
print(f"Đơn vị tiền tệ: {result.currency} (Kiểu dữ liệu: {type(result.currency)})")

--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---

--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---
{
    "vendor": "Circle K Việt Nam",
    "date": "2026-06-15",
    "total_amount": 47000.0,
    "currency": "VND"
}

--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---
Số tiền trích xuất: 47000.0 (Kiểu dữ liệu: <class 'float'>)
Đơn vị tiền tệ: VND (Kiểu dữ liệu: <class 'str'>)


In [33]:
# Giả lập một cục text cực kỳ lộn xộn, lỗi font, sai chính tả do tầng OCR quét từ ảnh sang
mock_ocr_output = """
--- KẾT QUẢ OCR HÓA ĐƠN HOÀN CHỈNH ---
📝 Hộ KINH DOANH CƠM GA NHA TRANG 2 CHJ EM ĐC 09 Thich MInh Nguyet; Phường Tân Sơn Hòa, TPHCM MST: 082089009531 HOTLINE: 0906619578
📝 Bàn 8
📝 PHIẾU TÍNH TIỀN HD035530
📝 'Giờ vào 07/06/2026 19.57
📝 Giờ in: 20,17 Thành Tên hàng Đ.Giá SL tien chương trinh khuyến mãi Coca Od 0 x 3 Cơm gà ta xé lớn (Phần) 58,000 x 2 116,000 Cơm xá xíu (Phần) 59,000 x 59,000 Sất trứng thêm (Phần) 7,000 7,000 Canh soup trứng tlem (Phần) 5,000 X 5,000 Tông ti`n hàng: 187,000 Chiết khấu: 187,000 TWng Cộng: Chuyen khoản Cảm ơn quý khách và hen gặp lai III Powered by KiotViet
"""

print("--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---")

# Gọi hàm xử lý
result = process_ocr_with_slm(mock_ocr_output)

print("\n--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---")
# Dùng model_dump() thay cho dict() để không bị báo lỗi cảnh báo màu vàng nữa nhé bạn
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=4))

print("\n--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---")
print(f"Số tiền trích xuất: {result.total_amount} (Kiểu dữ liệu: {type(result.total_amount)})")
print(f"Đơn vị tiền tệ: {result.currency} (Kiểu dữ liệu: {type(result.currency)})")

--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---

--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---
{
    "vendor": "Cơm Gà Nha Trang 2 Chị Em",
    "date": "2026-06-07",
    "total_amount": 187000.0,
    "currency": "VND"
}

--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---
Số tiền trích xuất: 187000.0 (Kiểu dữ liệu: <class 'float'>)
Đơn vị tiền tệ: VND (Kiểu dữ liệu: <class 'str'>)
